# T12 · demo local de actividad y salidas GRU

Carga los pesos, dibuja la actividad neuronal de una frase y predice sus fonemas con la salida elegida. Funciona en CPU con dos hilos.

La configuración está en `configs/speech_demo_salidas.yaml`. Por defecto utiliza la red conjunta de 200 pasos, run `xz71t4vd`, con salidas 2/3/5 y la primera frase de validación de la sesión configurada. Guarda los archivos de la demo en `runs/demo_salidas/`.


In [ ]:
from pathlib import Path
import os
import sys
import yaml

PROJECT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                if (path / "run.py").is_file() and (path / "src/models/speech_gru.py").is_file()), None)
if PROJECT is None:
    raise RuntimeError("Abre este notebook desde el repositorio del proyecto.")
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
CONFIG = Path("configs/speech_demo_salidas.yaml")
cfg = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[name] = str(cfg["ejecucion"]["hilos_cpu"])

os.environ["CUDA_VISIBLE_DEVICES"] = ""

from IPython.display import Image, display
from scripts.demo_speech_profiles import load_demo, create_controls
print("Python:", sys.executable)
print("Configuración:", CONFIG)


## Cargar los pesos y la frase

Recupera la arquitectura, el vocabulario y la normalización desde el checkpoint y comprueba su compatibilidad con los datos preparados. La frase queda fijada por la sesión, el split y la posición indicados en el YAML.


In [ ]:
if "demo" in globals():
    demo.close()
demo = load_demo(cfg)
print("Checkpoint:", demo.metadata["checkpoint"])
print("Paso guardado:", demo.metadata["checkpoint_step"])
print("Salidas disponibles:", demo.available_exits)
print("Dispositivo:", demo.metadata["hardware"])
print("Split:", demo.metadata["split"], "· sesión:", demo.metadata["session"],
      "· ensayo original:", demo.metadata["source_trial_index"])
print("Referencia:", demo.metadata["reference_text"])
print("Salidas de esta demo:", demo.output)


## Dibujar la actividad neuronal

Muestra `tx1` y `spikePow` de 6v normalizados con train y suavizados: la entrada común de las salidas del modelo. Los límites de color solo afectan al dibujo.


In [ ]:
activity = demo.draw_activity()
display(Image(filename=str(activity)))
print("Entrada:", demo.metadata["input_shape"], "· duración:", f"{demo.metadata['duration_seconds']:.2f} s")


## Predecir los fonemas

El selector ofrece las salidas disponibles en el checkpoint. La predicción muestra los fonemas, el PER de la frase incluyendo SIL y las capas GRU ejecutadas.

El tiempo mostrado es la mediana del forward con la entrada preparada, tras el calentamiento configurado.


In [ ]:
controls = create_controls(demo)
display(controls)


## Guardar las predicciones y cerrar la demo

Cada perfil guarda su predicción y procedencia en JSON. `demo.predict(capa)` permite obtenerla desde Python y `demo.close()` libera los recursos de la demo.
